!curl -LsSf https://astral.sh/uv/install.sh | sh
!test -d OpenDDE || git clone https://github.com/aurekaresearch/OpenDDE.git
%cd OpenDDE
!uv pip install --torch-backend cu126 -e '.[gpu]'
!uv pip install --group dev
!opendde doctor

In [ ]:
# General-purpose checkpoint used by default with -n opendde_v1.
%env OPENDDE_ROOT_DIR=/content/data
!mkdir -p $OPENDDE_ROOT_DIR/checkpoint

In [ ]:
%%bash
sudo apt remove curl
sudo apt purge curl
sudo apt-get update
sudo apt-get install -y libssl-dev autoconf libtool make
cd /usr/local/src
wget https://curl.haxx.se/download/curl-7.88.1.zip
unzip curl-7.88.1.zip
cd curl-7.88.1
./buildconf
./configure --with-ssl
make
sudo make install
sudo cp /usr/local/bin/curl /usr/bin/curl
sudo ldconfig
curl -V


Reading package lists...
Building dependency tree...
Reading state information...
Package 'curl' is not installed, so not removed
The following packages were automatically installed and are no longer required:
  libgoogle-perftools4 libtcmalloc-minimal4
Use 'sudo apt autoremove' to remove them.
0 upgraded, 0 newly installed, 0 to remove and 9 not upgraded.
Reading package lists...
Building dependency tree...
Reading state information...
Package 'curl' is not installed, so not removed
The following packages were automatically installed and are no longer required:
  libgoogle-perftools4 libtcmalloc-minimal4
Use 'sudo apt autoremove' to remove them.
0 upgraded, 0 newly installed, 0 to remove and 9 not upgraded.
Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRel





W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
--2026-07-13 23:29:10--  https://curl.haxx.se/download/curl-7.88.1.zip
Resolving curl.haxx.se (curl.haxx.se)... 151.101.2.49, 151.101.66.49, 151.101.130.49, ...
Connecting to curl.haxx.se (curl.haxx.se)|151.101.2.49|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://curl.se/download/curl-7.88.1.zip [following

In [ ]:
# The download script needs the `zstd` CLI (not the Python `zstd` package) to
# decompress the search-database .zst files, or it silently falls back to a
# broken S3 mirror (403).
!apt-get -qq update && apt-get -qq install -y zstd
!bash /content/OpenDDE/scripts/download_opendde_data.sh
# If you don't need MSA/template/RNA-MSA features (as in the pred command below),
# add --skip-search-database to skip several GB of files you won't use:
# !bash /content/OpenDDE/scripts/download_opendde_data.sh --skip-search-database

In [ ]:
# Smoke test with the minimal example from the OpenDDE README, to confirm
# the install + data download actually work before running a real job.
tiny_json = '''[
    {
        "name": "tiny",
        "modelSeeds": [101],
        "sequences": [
            {
                "proteinChain": {
                    "sequence": "ACDEFGHIK",
                    "count": 1
                }
            }
        ]
    }
]'''
with open("tiny.json", "w") as f:
    f.write(tiny_json)

!opendde pred -i tiny.json -o ./output -n opendde_v1 --use_msa false --use_template false --use_rna_msa false --sample 1 --step 200 --cycle 10

In [ ]:
# Your real job. Upload/create egfr-rad51-kinase.json in this working directory
# first (it wasn't part of the original notebook, so it must exist before running).
!opendde pred -i egfr-rad51-kinase.json -o ./output -n opendde_v1 --use_msa false --use_template false --use_rna_msa false --sample 1 --step 200 --cycle 10